# MiniMax H3｜ComfyUI 基礎版（Colab）

在 Colab 上裝好 ComfyUI 與 MiniMax H3 模型組，**不帶任何工作流**——開好介面後自己拉節點、或拖入自己的工作流學習。
由上往下依序執行。**執行階段請選 L4 GPU**（A100 也可以但很耗點數；T4 跑不動）。

| 步驟 | 做什麼 | 時間（L4 實測） |
|---|---|---|
| ① 檢查＋安裝 | GPU／驅動不符就停下，不安裝；通過才裝 ComfyUI＋節點管理器＋ListHelper | 約 1 分鐘 |
| ② 啟動 | 背景啟動 ComfyUI | 約 1 分鐘 |
| ③ 開介面 | Colab 內建代理（推薦）；③-b 是 cloudflare 備援 | 數秒 |
| ④ 下載模型 | 約 37 GB（ComfyUI 確定正常才下載） | 約 1 分鐘 |

想直接做「會說話的數字人」：改開同一個 repo 的 `H3_Digital_Human.ipynb`。

**Hugging Face 權杖（建議）**：左側欄「🔑 密鑰」新增 `HF_TOKEN`（請用**唯讀**權杖），並開啟「筆記本存取權」。

模型：`Minimax-h3_Singularity_x_10Eros-Max_PRISM`（int8，加速已融合，6 步）＋官方 Qwen3-VL 文字編碼器與 VAE。授權依照各來源模型。

最新版：<https://github.com/dseditor/minimax-h3-colab>

In [ ]:
#@title ① 檢查環境＋安裝 ComfyUI（GPU 或驅動不符就停下，不會安裝）
FIX_SRC = "# Colab 內建代理會把網址裡的 %2F 解成 /，ComfyUI 的 /userdata/{file} 就配不到 → 工作流清單點開 404。\n# 這個小節點補一條「被拆成兩段」的路由，讀寫同一個使用者資料夾。\nimport os\nfrom aiohttp import web\nimport folder_paths\nfrom server import PromptServer\n\nroutes = PromptServer.instance.routes\nBASE = os.path.abspath(os.path.join(folder_paths.get_user_directory(), 'default'))\n\n\ndef _path(req):\n    p = os.path.abspath(os.path.join(BASE, req.match_info['sub'], req.match_info['file']))\n    if not p.startswith(BASE + os.sep):\n        raise web.HTTPForbidden()\n    return p\n\n\n@routes.get('/userdata/{sub}/{file}')\nasync def _get(req):\n    p = _path(req)\n    if not os.path.isfile(p):\n        return web.Response(status=404)\n    return web.FileResponse(p)\n\n\n@routes.post('/userdata/{sub}/{file}')\nasync def _post(req):\n    p = _path(req)\n    if os.path.exists(p) and req.query.get('overwrite', 'true') == 'false':\n        return web.Response(status=409, text='File already exists')\n    os.makedirs(os.path.dirname(p), exist_ok=True)\n    with open(p, 'wb') as fh:\n        fh.write(await req.read())\n    rel = os.path.relpath(p, BASE).replace(os.sep, '/')\n    if req.query.get('full_info', 'false') == 'true':\n        st = os.stat(p)\n        return web.json_response({'path': rel, 'size': st.st_size, 'modified': st.st_mtime})\n    return web.json_response(rel)\n\n\n@routes.delete('/userdata/{sub}/{file}')\nasync def _delete(req):\n    p = _path(req)\n    if not os.path.isfile(p):\n        return web.Response(status=404)\n    os.remove(p)\n    return web.Response(status=204)\n\n\nNODE_CLASS_MAPPINGS = {}\n"
import torch, shutil, subprocess, sys, os
from pathlib import Path
if not torch.cuda.is_available():
    raise RuntimeError('沒有 GPU：請到「執行階段 → 變更執行階段類型」選 L4 GPU，再重跑這格')
cap = torch.cuda.get_device_capability(0)
drv = subprocess.run(['nvidia-smi', '--query-gpu=driver_version', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip()
print('GPU:', torch.cuda.get_device_name(0), '| capability', cap, '| VRAM', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), 'GiB')
print('驅動:', drv, '| 目前 PyTorch CUDA:', torch.version.cuda, '| 可用磁碟', round(shutil.disk_usage('/content').free / 2**30, 1), 'GiB')
if cap < (8, 0):
    raise RuntimeError(f'這張卡（capability {cap}）跑不動 int8 模型，沒有安裝任何東西。請換 L4（或 A100）再重跑。')
if int(drv.split('.')[0]) < 580:
    raise RuntimeError(f'驅動 {drv} 不支援 CUDA 13（需要 580 以上），沒有安裝任何東西。請換一台執行階段（L4）再試。')
print('✅ 環境檢查通過，開始安裝')
# HF 權杖先放進環境變數：之後啟動的 ComfyUI（含 ListHelper 的模型下載節點）都會沿用
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN'); print('已讀到 HF_TOKEN')
except Exception:
    print('沒有設定 HF_TOKEN，之後改用匿名下載（人多時可能被限速）')
COMFY = Path('/content/ComfyUI')
if not COMFY.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Comfy-Org/ComfyUI.git', str(COMFY)], check=True)
if not (torch.version.cuda or '').startswith('13'):
    print('安裝 CUDA 13 版 PyTorch（int8 ConvRot 權重需要）…')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch', 'torchvision', 'torchaudio',
                    '--index-url', 'https://download.pytorch.org/whl/cu130'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(COMFY / 'requirements.txt'), 'huggingface_hub', 'hf_xet'], check=True)
# 預裝 ListHelper 節點組（含 ModelDownloader：在介面裡從 Hugging Face 抓模型）
LH = COMFY / 'custom_nodes' / 'ComfyUI-ListHelper'
if not LH.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/dseditor/ComfyUI-ListHelper.git', str(LH)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(LH / 'requirements.txt')], check=True)
# Colab 內建代理（③）會把網址的 %2F 解開，工作流清單點開會 404 → 裝一個補路由的小節點
FIX = COMFY / 'custom_nodes' / 'colab_userdata_slash_fix'
FIX.mkdir(parents=True, exist_ok=True)
(FIX / '__init__.py').write_text(FIX_SRC, encoding='utf-8')
# 節點管理器（ComfyUI-Manager 已併入官方，用 pip 套件＋啟動旗標 --enable-manager）
if (COMFY / 'manager_requirements.txt').exists():
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(COMFY / 'manager_requirements.txt')], check=True)
out = subprocess.run([sys.executable, '-c', 'import torch;print(torch.__version__, torch.version.cuda, torch.cuda.is_available())'],
                     capture_output=True, text=True)
print('ComfyUI 會用的 PyTorch：', out.stdout.strip(), out.stderr[-300:])


In [ ]:
#@title ② 啟動 ComfyUI（背景執行）
import subprocess, time, urllib.request, sys
# --enable-cors-header：經 cloudflare 通道開介面時，瀏覽器的 Origin（trycloudflare 網址）
# 跟轉進來的 Host（127.0.0.1）對不上，ComfyUI 預設會回 403「存取遭拒」；開這個旗標才放行。
if 'server' in globals() and server.poll() is None:
    server.terminate(); server.wait(); print('已關閉舊的 ComfyUI，重新啟動')
# --enable-compress-response-body：介面要抓一大包節點清單和前端檔案，壓縮後走通道快很多。
# 節點管理器安全等級：normal- ＝可從清單裝節點，也可用 Git 網址安裝（本機模式下）。
# 這台 Colab 用完即丟；但 HF_TOKEN 在環境變數裡，自訂節點讀得到 → HF 權杖請用「唯讀」的。
cfg = COMFY / 'user' / '__manager' / 'config.ini'
cfg.parent.mkdir(parents=True, exist_ok=True)
if not cfg.exists():
    cfg.write_text('[default]\nsecurity_level = normal-\n', encoding='utf-8')
log = open('/content/comfyui.log', 'w')
server = subprocess.Popen([sys.executable, 'main.py', '--listen', '127.0.0.1', '--port', '8188', '--enable-cors-header', '--enable-manager', '--enable-compress-response-body'],
                          cwd=str(COMFY), stdout=log, stderr=subprocess.STDOUT)
for i in range(180):
    try:
        urllib.request.urlopen('http://127.0.0.1:8188/system_stats', timeout=2)
        print('ComfyUI 已啟動'); break
    except Exception:
        time.sleep(2)
else:
    print(open('/content/comfyui.log').read()[-3000:])
    raise RuntimeError('ComfyUI 沒有啟動成功，看上面的紀錄')


In [ ]:
#@title ③ 開啟 ComfyUI 介面（推薦：Colab 內建代理，快、穩、網址不外流）
# 執行後下面會出現一個連結；顯示成 localhost 也沒關係，直接點它就會開新分頁。
# ⚠️ 開分頁的瀏覽器必須登入「跟這本 Colab 同一個 Google 帳號」；帳號不同就改用下一格 ③-b。
from google.colab import output
output.serve_kernel_port_as_window(8188)


In [ ]:
#@title ③-b（備援）用 cloudflare 網址開介面：瀏覽器登入的 Google 帳號和 Colab 不同時用這格
# 會產生一個 trycloudflare 網址，點開就是 ComfyUI；也是交給桌面版 AI 操作時要給的網址。
# ⚠️ 網址沒有密碼，拿到的人都能用你的 GPU —— 不要貼到公開的地方。
import subprocess, re, time, json, urllib.request
if not Path('/content/cloudflared').exists():
    subprocess.run(['wget', '-q', '-O', '/content/cloudflared',
                    'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'], check=True)
    subprocess.run(['chmod', '+x', '/content/cloudflared'], check=True)
# 通道的輸出寫進檔案，不要接 PIPE：PIPE 沒人讀會塞滿，通道就卡住，網頁停在載入畫面
if 'tun' in globals() and tun.poll() is None:
    tun.terminate(); tun.wait()
open('/content/cloudflared.log', 'w').close()
tun = subprocess.Popen(['/content/cloudflared', 'tunnel', '--no-autoupdate', '--url', 'http://127.0.0.1:8188'],
                       stdout=open('/content/cloudflared.log', 'a'), stderr=subprocess.STDOUT)
url = None
for i in range(60):
    time.sleep(1)
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('/content/cloudflared.log').read())
    if m:
        url = m.group(0); break
print('👉 ComfyUI 介面：', url or '（沒拿到網址，看 /content/cloudflared.log）')


In [ ]:
#@title ④ 下載模型（約 37 GB）
# 預設直接從 Hugging Face 下載（最快）。HF 權杖已在 ① 讀入。
# 只有在 Hugging Face 撞到流量上限時，才勾選 USE_DRIVE_CACHE：
#   勾選後會從「我的雲端硬碟/H3_models」複製（比 HF 慢），雲端硬碟沒有的檔案才去 HF 抓並順便存一份。
USE_DRIVE_CACHE = False  #@param {type:"boolean"}
from huggingface_hub import hf_hub_download
import shutil, os
CACHE = None
if USE_DRIVE_CACHE:
    from google.colab import drive
    drive.mount('/content/drive')
    CACHE = Path('/content/drive/MyDrive/H3_models'); CACHE.mkdir(parents=True, exist_ok=True)
files = [
    ('OliviaRossi/Minimax-h3_Singularity-x-10Eros-Max_PRISM', 'Minimax-h3_Singularity_x_10Eros-Max_PRISM_int8.safetensors', 'diffusion_models'),
    ('Comfy-Org/MiniMax-H3', 'text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors', 'text_encoders'),
    ('Comfy-Org/MiniMax-H3', 'vae/minimax_h3_video_vae_int8_convrot.safetensors', 'vae'),
    ('Comfy-Org/MiniMax-H3', 'vae/minimax_h3_audio_vae_fp32.safetensors', 'vae'),
]
for repo, name, folder in files:
    target = COMFY / 'models' / folder / Path(name).name
    target.parent.mkdir(parents=True, exist_ok=True)
    cached = CACHE / Path(name).name if CACHE else None
    if not target.is_file():
        if cached and cached.is_file():
            print('從雲端硬碟複製', cached.name); shutil.copy(str(cached), str(target))
        else:
            got = Path(hf_hub_download(repo_id=repo, filename=name, local_dir='/content/hf'))
            shutil.move(str(got), str(target))
    if cached and not cached.is_file():
        print('存一份到雲端硬碟', cached.name); shutil.copy(str(target), str(cached))
    print(folder, target.name, f'{target.stat().st_size / 2**30:.1f} GiB')


In [ ]:
#@title 🔍 出問題時跑這格：看 GPU 與 ComfyUI 紀錄
import subprocess
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
log = open('/content/comfyui.log', encoding='utf-8', errors='ignore').read()
keys = ('error', 'Error', 'int8', 'quant', 'fallback', 'out of memory', 'loaded', 'Prompt executed')
print('\n'.join(l for l in log.splitlines() if any(k in l for k in keys))[-4000:])
